# Albert's AI Song Studio — temporary GPU worker

This notebook runs the open-source ACE-Step 1.5 API for your own studio. Open it in Google Colab from an Android phone. A free GPU may not always be available, and Colab sessions are temporary.

**Security:** use a fresh, long random API key. Do not share the notebook output or tunnel URL publicly. When the session ends, the worker stops.


## 1. Select a GPU
In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. Free GPU access is subject to availability and account limits.


In [ ]:
import torch, os, subprocess, sys, textwrap, time, getpass
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Select a GPU runtime before continuing.')


## 2. Install ACE-Step 1.5
This downloads the model code and dependencies. It can take several minutes and requires substantial temporary disk space.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Select a GPU runtime and rerun this cell.')
%cd /content
if not os.path.isdir('/content/ACE-Step-1.5'):
    !git clone https://github.com/ACE-Step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!pip -q install uv
!uv sync


## 3. Configure the private API key
Create a long random key. You will later enter the same key into the private server environment for Albert's web app. Never put it in a public GitHub commit.


In [ ]:
import secrets
API_KEY = secrets.token_urlsafe(32)
print('Generated API key (copy it somewhere private):')
print(API_KEY)


## 4. Start the ACE-Step API
This starts the 2B Turbo DiT engine without the optional language model. Use your own lyrics or instrumental prompts in this first test. Auto-generated lyrics need a compatible 5Hz language model and more GPU memory.


In [ ]:
os.environ['ACESTEP_API_HOST'] = '0.0.0.0'
os.environ['ACESTEP_API_PORT'] = '8001'
os.environ['SERVER_NAME'] = '0.0.0.0'
os.environ['PORT'] = '8001'
os.environ['ACESTEP_CONFIG_PATH'] = 'acestep-v15-turbo'
os.environ['ACESTEP_INIT_LLM'] = 'false'
os.environ['ACESTEP_API_KEY'] = API_KEY
log = open('/content/acestep-api.log','w')
api_process = subprocess.Popen(['uv','run','acestep-api'], cwd='/content/ACE-Step-1.5', env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)
time.sleep(20)
print('ACE-Step process running:', api_process.poll() is None)
print('Logs: !tail -n 60 /content/acestep-api.log')
if api_process.poll() is not None:
    raise RuntimeError('ACE-Step exited during startup. Inspect /content/acestep-api.log')


## 5. Create a temporary HTTPS tunnel
The tunnel URL changes when you restart the notebook. Keep this notebook session open while generating songs. The API key protects requests; do not share the URL and key together.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
tunnel_log = open('/content/cloudflared.log','w')
tunnel_process = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8001','--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
time.sleep(8)
import re
log_text = open('/content/cloudflared.log').read()
match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
if not match:
    print(log_text[-3000:])
    raise RuntimeError('Tunnel URL not found yet. Wait a few seconds and rerun this cell.')
ENGINE_URL = match.group(0)
print('ACE-Step API URL (copy privately):', ENGINE_URL)
print('API key:', API_KEY)


## 6. Health check
If this returns a healthy response, the cloud engine is running. Configure your studio server with `ACESTEP_API_URL` set to the tunnel URL and `ACESTEP_API_KEY` set to the generated key. Do not use a `NEXT_PUBLIC_` variable for the key.


In [ ]:
import requests
try:
    response = requests.get(ENGINE_URL + '/health', headers={'Authorization':'Bearer '+API_KEY}, timeout=20)
    print('HTTP', response.status_code)
    print(response.text[:2000])
except Exception as exc:
    print('Health check could not complete:', repr(exc))
    print('Check /content/acestep-api.log and /content/cloudflared.log')


## Important operating notes
- Keep this Colab tab/session running while songs are being generated.
- Free GPU availability, runtime duration, and quotas are controlled by Colab and can change.
- The first model download may take time. A disconnected session may need to download it again.
- This is a development tunnel, not a permanent production endpoint.
- Never publish your API key, tunnel URL, or notebook output.
- When finished, stop the runtime to shut down the worker and tunnel.

Official engine: https://github.com/ACE-Step/ACE-Step-1.5
